# 🏰 TrOCR + LoRA — Fine-tuning sur CREMMA Médiéval

Fine-tuning de `microsoft/trocr-base-handwritten` avec LoRA (r=8) sur des manuscrits médiévaux français (XIIIe-XVe siècle).

**⚠️ Activer GPU T4 x2 dans Session Options**

## 1. Installation

In [ ]:
%%capture
!pip install numpy==1.26.4
!pip install scipy==1.13.1 scikit-learn==1.5.2
!pip install transformers==4.40.0
!pip install peft==0.10.0
!pip install datasets==2.19.0
!pip install editdistance==0.8.1
!pip install Pillow>=10.0
!pip install accelerate>=0.26.0
!pip install sentencepiece
!pip install lxml

In [ ]:
import os, json, glob, random, gc
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import editdistance

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

def fix_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

fix_seeds(42)
OUTPUT_DIR = '/kaggle/working/models/trocr-cremma-lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')

## 2. Données CREMMA

In [ ]:
!git clone https://github.com/HTR-United/cremma-medieval data/cremma 2>/dev/null || echo 'Already cloned'
print('✅ CREMMA ready')

In [ ]:
import xml.etree.ElementTree as ET
from sklearn.model_selection import GroupShuffleSplit

lines_dir = Path('data/lines')
lines_dir.mkdir(parents=True, exist_ok=True)

all_records = []
cremma_dir = Path('data/cremma/data')
line_counter = 0

for ms_dir in sorted(cremma_dir.iterdir()):
    if not ms_dir.is_dir():
        continue
    ms_name = ms_dir.name
    for xml_path in sorted(f for f in ms_dir.glob('*.xml') if 'chocomufin' not in f.name):
        img_path = xml_path.with_suffix('.jpg')
        if not img_path.exists():
            continue
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
        except:
            continue
        ns = root.tag.split('}')[0] + '}' if root.tag.startswith('{') else ''
        try:
            page_img = Image.open(img_path).convert('RGB')
        except:
            continue
        for i, text_line in enumerate(root.iter(f'{ns}TextLine')):
            parts = [s.get('CONTENT', '') for s in text_line.iter(f'{ns}String') if s.get('CONTENT')]
            text = ' '.join(parts).strip()
            if len(text) < 2:
                continue
            try:
                hpos, vpos = float(text_line.get('HPOS', 0)), float(text_line.get('VPOS', 0))
                width, height = float(text_line.get('WIDTH', 0)), float(text_line.get('HEIGHT', 0))
            except:
                continue
            if width <= 0 or height <= 0:
                continue
            x0, y0 = max(0, int(hpos)), max(0, int(vpos))
            x1, y1 = min(page_img.width, int(hpos+width)), min(page_img.height, int(vpos+height))
            if x1 <= x0 or y1 <= y0:
                continue
            line_img_path = lines_dir / f'{line_counter:05d}.png'
            page_img.crop((x0, y0, x1, y1)).save(line_img_path)
            all_records.append({'img_path': str(line_img_path), 'text': text, 'manuscript': ms_name})
            line_counter += 1

print(f'✅ {len(all_records)} lines extracted')

# Split by manuscript
manuscripts_list = [r['manuscript'] for r in all_records]
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(all_records, groups=manuscripts_list))
train_records = [all_records[i] for i in train_idx]
val_records = [all_records[i] for i in val_idx]

print(f'✅ Train: {len(train_records)}, Val: {len(val_records)}')
print(f'✅ No data leakage: {len(set(r["manuscript"] for r in train_records) & set(r["manuscript"] for r in val_records)) == 0}')

## 3. Charger TrOCR + LoRA

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model

MODEL_NAME = 'microsoft/trocr-base-handwritten'
print(f'Chargement de {MODEL_NAME}...')
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
print(f'Paramètres totaux: {model.num_parameters():,}')

In [ ]:
# LoRA r=8
LORA_R = 8

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 4,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model = model.to(device)
print(f'Modèle sur {device}')

## 4. Dataset + Métriques

In [ ]:
MAX_LENGTH = 128

class LazyHTRDataset(torch.utils.data.Dataset):
    def __init__(self, records, processor, max_length=MAX_LENGTH):
        self.processor = processor
        self.max_length = max_length
        self.img_paths = [r['img_path'] for r in records]
        self.texts = [r['text'] for r in records]
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        pixel_values = self.processor(img, return_tensors='pt').pixel_values[0]
        labels = self.processor.tokenizer(
            self.texts[idx], padding='max_length', max_length=self.max_length,
            truncation=True, return_tensors='pt'
        ).input_ids[0]
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}

train_dataset = LazyHTRDataset(train_records, processor)
val_dataset = LazyHTRDataset(val_records, processor)

del all_records
gc.collect()
print(f'✅ Train: {len(train_dataset)}, Val: {len(val_dataset)}')

In [ ]:
def compute_cer(predictions, references):
    total_errors = sum(editdistance.eval(p, r) for p, r in zip(predictions, references))
    total_chars = sum(len(r) for r in references)
    return total_errors / total_chars if total_chars > 0 else 0.0

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    return {'cer': compute_cer(pred_str, label_str)}

## 5. Entraînement

In [ ]:
EPOCHS = 30
BATCH_SIZE = 8
LEARNING_RATE = 5e-5

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    fp16=True,
    seed=42,
    logging_steps=25,
    logging_dir=f'{OUTPUT_DIR}/logs',
    report_to='none',
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    save_total_limit=3,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print(f'✅ Trainer ready — Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}, LoRA r={LORA_R}')

In [ ]:
%%time
# 🚀 Train (resumes from checkpoint if exists)
print('=' * 60)
print('🚀 Fine-tuning TrOCR + LoRA r=8')
print('=' * 60)

checkpoints = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))
if checkpoints:
    print(f'📂 Resuming from: {checkpoints[-1]}')
    train_result = trainer.train(resume_from_checkpoint=checkpoints[-1])
else:
    print('🆕 Training from scratch')
    train_result = trainer.train()

print(f'\n✅ Done! Loss: {train_result.training_loss:.4f}')

In [ ]:
# Sauvegarder
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f'✅ Model saved: {OUTPUT_DIR}')

## 6. Évaluation

In [ ]:
eval_results = trainer.evaluate()

print('=' * 60)
print('📊 RÉSULTATS TrOCR + LoRA r=8')
print('=' * 60)
print(f"CER: {eval_results['eval_cer']:.4f} ({eval_results['eval_cer']*100:.1f}%)")
print(f"Loss: {eval_results['eval_loss']:.4f}")
print('=' * 60)
if eval_results['eval_cer'] < 0.10:
    print('\n🎉 CER < 10% — Objectif atteint!')
else:
    print(f"\n📈 CER = {eval_results['eval_cer']*100:.1f}%")

In [ ]:
# Exemples qualitatifs
model.eval()
sample_indices = random.sample(range(len(val_records)), min(10, len(val_records)))

preds_sample, refs_sample = [], []
for idx in sample_indices:
    record = val_records[idx]
    img = Image.open(record['img_path']).convert('RGB')
    pixel_values = processor(img, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_new_tokens=128)
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ref_text = record['text']
    preds_sample.append(pred_text)
    refs_sample.append(ref_text)
    line_cer = editdistance.eval(pred_text, ref_text) / max(len(ref_text), 1)
    marker = '✓' if line_cer < 0.10 else '✗'
    print(f"{marker} REF:  {ref_text[:70]}")
    print(f"  PRED: {pred_text[:70]}")
    print(f"  CER:  {line_cer:.1%}\n")

print(f'\nCER échantillon: {compute_cer(preds_sample, refs_sample):.1%}')

## 7. Résumé

In [ ]:
print('=' * 60)
print('📊 RÉSUMÉ')
print('=' * 60)
print(f'Modèle: TrOCR Base + LoRA r={LORA_R}')
print(f'Corpus: CREMMA Médiéval ({len(train_records)} train, {len(val_records)} val)')
print(f'Zero-shot CER: 67.9%')
if 'eval_results' in dir():
    print(f"Fine-tuned CER: {eval_results['eval_cer']*100:.1f}%")
    print(f"Amélioration: {67.9 - eval_results['eval_cer']*100:.1f} points")
print(f'\n💾 Modèle: {OUTPUT_DIR}')
print('   → Télécharger depuis Output tab')